In [6]:
from google.colab import drive;
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os

# List everything in your Google Drive root
print(os.listdir('/content/drive/MyDrive/'))


['Untitled document (1).gdoc', 'Untitled document.gdoc', 'MVCS-Revision.gdoc', '[Mẫu] Tiểu luận môn LSĐ CSVN.gdoc', 'MẪU TIỂU LUẬN MÔN TƯ TƯỞNG HỒ CHÍ MINH.gdoc', 'Colab Notebooks', 'DL4AI']


In [10]:
print(os.listdir('/content/drive/MyDrive/DL4AI/'))


['nasdag', 'Vietnam dataset']


In [11]:
data_dir = '/content/drive/MyDrive/DL4AI/nasdag/'

files = os.listdir(data_dir)
csv_files = [f for f in files if f.endswith('.csv')]
print(f"Total CSV files: {len(csv_files)}")
print("Sample tickers:", csv_files[:5])


Total CSV files: 0
Sample tickers: []


In [12]:
# List everything (not just CSVs)
all_files = os.listdir('/content/drive/MyDrive/DL4AI/nasdag/')
print(f"Total items: {len(all_files)}")
print("First 10 items:", all_files[:10])


Total items: 1
First 10 items: ['csv']


In [13]:
data_dir = '/content/drive/MyDrive/DL4AI/nasdag/csv/'

# Verify
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
print(f"Total CSV files: {len(csv_files)}")
print("Sample tickers:", csv_files[:5])


Total CSV files: 1564
Sample tickers: ['FTEK.csv', 'FBP.csv', 'ADUS.csv', 'TOWN.csv', 'DLTR.csv']


In [14]:
# Set correct path + get tickers
data_dir = '/content/drive/MyDrive/DL4AI/nasdag/csv/'
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
your_tickers = set(f.replace('.csv','') for f in csv_files)

# Match with S&P500 sectors
matched = sp500[sp500['Symbol'].isin(your_tickers)]
print(f"Matched: {len(matched)} tickers")
print(matched['GICS Sector'].value_counts())

# Get Technology tickers
tech_tickers = matched[matched['GICS Sector'] == 'Information Technology']['Symbol'].tolist()
print(f"\nTechnology tickers: {len(tech_tickers)}")
print(tech_tickers)


Matched: 110 tickers


KeyError: 'GICS Sector'

In [15]:
import pandas as pd
import os

# Step 1: Reload S&P500 list
url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
sp500 = pd.read_csv(url)
print("S&P500 columns:", sp500.columns.tolist())

# Step 2: Get your tickers
data_dir = '/content/drive/MyDrive/DL4AI/nasdag/csv/'
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
your_tickers = set(f.replace('.csv','') for f in csv_files)
print(f"Your tickers: {len(your_tickers)}")

# Step 3: Match
matched = sp500[sp500['Symbol'].isin(your_tickers)]
print(f"Matched: {len(matched)}")
print(matched['GICS Sector'].value_counts())

# Step 4: Technology tickers
tech_tickers = matched[matched['GICS Sector'] == 'Information Technology']['Symbol'].tolist()
print(f"\nTech tickers: {len(tech_tickers)}")
print(tech_tickers)


S&P500 columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']
Your tickers: 1564
Matched: 110
GICS Sector
Information Technology    37
Financials                15
Health Care               14
Consumer Discretionary    14
Industrials               13
Communication Services     5
Real Estate                4
Consumer Staples           4
Energy                     2
Materials                  2
Name: count, dtype: int64

Tech tickers: 37
['ADBE', 'AMD', 'AKAM', 'ADI', 'AAPL', 'AMAT', 'ADSK', 'CDNS', 'CDW', 'CSCO', 'CTSH', 'EPAM', 'FFIV', 'FSLR', 'FTNT', 'INTC', 'INTU', 'LRCX', 'MCHP', 'MU', 'MSFT', 'MPWR', 'NTAP', 'NVDA', 'NXPI', 'ON', 'PTC', 'QCOM', 'STX', 'SWKS', 'SMCI', 'SNPS', 'TXN', 'TYL', 'VRSN', 'WDC', 'ZBRA']


In [17]:
# Check actual columns of one ticker
sample_ticker = tech_tickers[0]
df_test = pd.read_csv(f'{data_dir}{sample_ticker}.csv')
print(f"Ticker: {sample_ticker}")
print(f"Columns: {df_test.columns.tolist()}")
print(df_test.head(3))


Ticker: ADBE
Columns: ['Date', 'Low', 'Open', 'Volume', 'High', 'Close', 'Adjusted Close']
         Date       Low  Open    Volume      High     Close  Adjusted Close
0  13-08-1986  0.210938   0.0  18899200  0.218750  0.210938        0.198057
1  14-08-1986  0.222656   0.0   4160000  0.230469  0.222656        0.209060
2  15-08-1986  0.218750   0.0   4332800  0.222656  0.218750        0.205392


In [19]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

FEATURES = ['Open', 'High', 'Low', 'Close', 'Adjusted Close', 'Volume']
MIN_ROWS = 500

def load_filtered_stocks(tickers, data_dir, min_rows=MIN_ROWS):
    stocks = {}
    for ticker in tickers:
        path = os.path.join(data_dir, f'{ticker}.csv')
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').set_index('Date')
        df = df[FEATURES].dropna()
        if len(df) >= min_rows:
            stocks[ticker] = df
    return stocks

tech_stocks = load_filtered_stocks(tech_tickers, data_dir, min_rows=MIN_ROWS)


# Preview each ticker
for ticker, df in tech_stocks.items():
    print(f"{ticker:6s} | Rows: {len(df):5d} | {df.index[0].date()} → {df.index[-1].date()}")


/tmp/ipython-input-2342595569.py:14: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])
/tmp/ipython-input-2342595569.py:14: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])
/tmp/ipython-input-2342595569.py:14: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])
/tmp/ipython-input-2342595569.py:14: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])


ValueError: time data "15-12-1980" doesn't match format "%m-%d-%Y", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [20]:
FEATURES = ['Open', 'High', 'Low', 'Close', 'Adjusted Close', 'Volume']
TARGET = 'Open'
MIN_ROWS = 500

def load_filtered_stocks(tickers, data_dir, min_rows=MIN_ROWS):
    stocks = {}
    skipped = []
    for ticker in tickers:
        path = os.path.join(data_dir, f'{ticker}.csv')
        if not os.path.exists(path):
            skipped.append((ticker, 'file not found'))
            continue
        try:
            df = pd.read_csv(path)
            # Handle mixed date formats safely
            df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
            df = df.dropna(subset=['Date'])  # drop rows with unparseable dates
            df = df.sort_values('Date').set_index('Date')
            
            # Check all features exist
            missing = [f for f in FEATURES if f not in df.columns]
            if missing:
                skipped.append((ticker, f'missing cols: {missing}'))
                continue
            
            df = df[FEATURES].dropna()
            if len(df) >= min_rows:
                stocks[ticker] = df
            else:
                skipped.append((ticker, f'only {len(df)} rows'))
        except Exception as e:
            skipped.append((ticker, str(e)))
    
    print(f"Loaded: {len(stocks)} stocks")
    print(f"Skipped: {len(skipped)}")
    for t, reason in skipped:
        print(f"   {t}: {reason}")
    return stocks

tech_stocks = load_filtered_stocks(tech_tickers, data_dir, min_rows=MIN_ROWS)

# Preview loaded stocks
for ticker, df in tech_stocks.items():
    print(f"{ticker:6s} | Rows: {len(df):5d} | {df.index[0].date()} → {df.index[-1].date()}")


Loaded: 36 stocks
Skipped: 1
   LRCX: Error tokenizing data. C error: Expected 7 fields in line 4926, saw 9

ADBE   | Rows:  9158 | 1986-08-13 → 2022-12-12
AMD    | Rows: 10778 | 1980-03-17 → 2022-12-12
AKAM   | Rows:  5818 | 1999-10-29 → 2022-12-12
ADI    | Rows: 10778 | 1980-03-17 → 2022-12-12
AAPL   | Rows: 10590 | 1980-12-12 → 2022-12-12
AMAT   | Rows: 10778 | 1980-03-17 → 2022-12-12
ADSK   | Rows:  9441 | 1985-06-28 → 2022-12-12
CDNS   | Rows:  8950 | 1987-06-10 → 2022-12-12
CDW    | Rows:  2383 | 2013-06-27 → 2022-12-12
CSCO   | Rows:  8269 | 1990-02-16 → 2022-12-12
CTSH   | Rows:  6162 | 1998-06-19 → 2022-12-12
EPAM   | Rows:  2730 | 2012-02-08 → 2022-12-12
FFIV   | Rows:  5921 | 1999-06-04 → 2022-12-12
FSLR   | Rows:  4044 | 2006-11-17 → 2022-12-12
FTNT   | Rows:  3289 | 2009-11-18 → 2022-12-12
INTC   | Rows: 10778 | 1980-03-17 → 2022-12-12
INTU   | Rows:  7494 | 1993-03-12 → 2022-12-12
MCHP   | Rows:  7489 | 1993-03-19 → 2022-12-12
MU     | Rows:  9713 | 1984-06-01 → 2022-12-1

In [3]:
import pandas as pd
import os

# Load S&P500 list
url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
sp500 = pd.read_csv(url)

# Check actual column names first
print("Columns:", sp500.columns.tolist())
print(sp500.head(3))


Columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']
  Symbol             Security  GICS Sector         GICS Sub-Industry  \
0    MMM                   3M  Industrials  Industrial Conglomerates   
1    AOS          A. O. Smith  Industrials         Building Products   
2    ABT  Abbott Laboratories  Health Care     Health Care Equipment   

     Headquarters Location  Date added    CIK Founded  
0    Saint Paul, Minnesota  1957-03-04  66740    1902  
1     Milwaukee, Wisconsin  2017-07-26  91142    1916  
2  North Chicago, Illinois  1957-03-04   1800    1888  


In [8]:
# Correct column names from your output
data_dir = '/content/drive/MyDrive/csv/'
your_tickers = set(f.replace('.csv','') for f in os.listdir(data_dir) if f.endswith('.csv'))

# Match with correct column name 'GICS Sector'
matched = sp500[sp500['Symbol'].isin(your_tickers)]
print(f"Matched: {len(matched)} tickers in your data")
print("\nAll sectors available:")
print(matched['GICS Sector'].value_counts())


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/csv/'